In [0]:
from pyspark.sql.functions import col, regexp_replace, to_date

file_path = "/Volumes/workspace/default/wallmart_sales_data/Walmart.csv" 
df = spark.read.csv(file_path, header=True, inferSchema=True)
bad_cols = ['11', '12', '_c11', '_c12']
df = df.drop(*[c for c in bad_cols if c in df.columns])

df = df.withColumn("unit_price", regexp_replace(col("unit_price"), "[\\$,]", "").cast("double"))
df = df.withColumn("quantity", col("quantity").cast("int"))

df = df.withColumn("total", col("unit_price") * col("quantity"))

df = df.withColumn("date", to_date(col("date"), "dd-MM-yyyy"))

df = df.dropDuplicates().dropna(subset=['invoice_id', 'total'])

df.createOrReplaceTempView("sales_data")

print(f"Schema fixed. Total rows loaded: {df.count()}")

Schema fixed. Total rows loaded: 9969


In [0]:
%sql
--1. Identify the most common payment method
SELECT payment_method, COUNT(*) as total_transactions
FROM sales_data
GROUP BY payment_method
ORDER BY total_transactions DESC;

--2. Which category sold the highest total quantity?

SELECT category, SUM(quantity) as total_qty
FROM sales_data
GROUP BY 1 ORDER BY 2 DESC LIMIT 1;

--3. Determine the total revenue by month

SELECT 
    month(date) as month_id,
    date_format(date, 'MMMM') as month_name,
    ROUND(SUM(total), 2) as monthly_revenue
FROM sales_data
GROUP BY 1, 2
ORDER BY 1;

--4. Which city has the highest average rating?
SELECT 
    City, 
    ROUND(AVG(rating), 2) as avg_rating
FROM sales_data
GROUP BY 1
ORDER BY 2 DESC
LIMIT 1;

--5. Find the total profit for each branch
SELECT 
    Branch, 
    ROUND(SUM(total * profit_margin), 2) as total_profit
FROM sales_data
GROUP BY 1
ORDER BY 2 DESC;

--6. Which day of the week has the highest sales volume (total revenue)?
SELECT 
    date_format(date, 'EEEE') as day_name,
    ROUND(SUM(total), 2) as total_sales
FROM sales_data
GROUP BY 1
ORDER BY 2 DESC
LIMIT 1;

-- 7. Revenue by Category (Simple and clean)
SELECT 
    category, 
    ROUND(SUM(total), 2) as total_revenue,
    ROUND(SUM(total * profit_margin), 2) as total_profit
FROM sales_data
GROUP BY 1
ORDER BY total_revenue DESC;

-- 8. Average Rating Analysis
SELECT 
    City, 
    category, 
    ROUND(AVG(rating), 2) as avg_rating
FROM sales_data
GROUP BY 1, 2
ORDER BY 1, 3 DESC;

-- 9. Monthly Sales Trend (Using the cleaned date)
SELECT 
    TRUNC(date, 'MM') as month,
    SUM(total) as monthly_revenue
FROM sales_data
GROUP BY 1
ORDER BY 1;

--10. Highest rated category per Branch and City 

WITH cat_rank AS (
    SELECT Branch, City, category, AVG(rating) as avg_rating,
    RANK() OVER (PARTITION BY Branch, City ORDER BY AVG(rating) DESC) as rnk
    FROM sales_data GROUP BY 1, 2, 3
)
SELECT Branch, City, category, ROUND(avg_rating, 2) FROM cat_rank WHERE rnk = 1;

--11. Branch with Highest Revenue Decrease Ratio (previous_year 2019,current_year 2020)
WITH yearly_sales AS (
    SELECT 
        Branch, 
        YEAR(date) as year_id, 
        SUM(total) as total_revenue
    FROM sales_data
    GROUP BY 1, 2
),
growth_comparison AS (
    SELECT 
        cur.Branch,
        prev.year_id as previous_year,
        cur.year_id as current_year,
        ROUND(prev.total_revenue, 2) as prev_rev,
        ROUND(cur.total_revenue, 2) as curr_rev,
        ROUND(((cur.total_revenue - prev.total_revenue) / prev.total_revenue) * 100, 2) as growth_pct
    FROM yearly_sales cur
    JOIN yearly_sales prev 
        ON cur.Branch = prev.Branch 
        AND cur.year_id = prev.year_id + 1
)
SELECT 
    Branch, 
    previous_year, 
    current_year, 
    prev_rev, 
    curr_rev, 
    growth_pct as decrease_ratio
FROM growth_comparison
WHERE growth_pct < 0
ORDER BY growth_pct ASC;


Branch,previous_year,current_year,prev_rev,curr_rev,decrease_ratio
WALM092,2019,2020,1477.9,213.0,-85.59
WALM014,2019,2020,2801.46,418.0,-85.08
WALM039,2019,2020,2188.33,367.0,-83.23
WALM049,2019,2020,5369.81,990.0,-81.56
WALM063,2019,2020,3939.08,737.0,-81.29
WALM071,2019,2020,3706.19,716.0,-80.68
WALM057,2019,2020,5891.74,1147.0,-80.53
WALM093,2019,2020,3082.01,622.0,-79.82
WALM070,2019,2020,4777.01,1033.0,-78.38
WALM095,2019,2020,6323.65,1495.0,-76.36
